In [39]:
import arviz as az
import IPython
from meridian import constants
from meridian.analysis import analyzer
from meridian.analysis import formatter
from meridian.analysis import optimizer
from meridian.analysis import summarizer
from meridian.analysis import visualizer
from meridian.data import data_frame_input_data_builder
from meridian.data import test_utils
from meridian.model import model
from meridian.model import prior_distribution
from meridian.model import spec
import numpy as np
import pandas as pd
# check if GPU is available
from psutil import virtual_memory
import tensorflow as tf
import tensorflow_probability as tfp

ram_gb = virtual_memory().total / 1e9
print('Your runtime has {:.1f} gigabytes of available RAM\n'.format(ram_gb))
print(
    'Num GPUs Available: ',
    len(tf.config.experimental.list_physical_devices('GPU')),
)
print(
    'Num CPUs Available: ',
    len(tf.config.experimental.list_physical_devices('CPU')),
)

Your runtime has 25.8 gigabytes of available RAM

Num GPUs Available:  0
Num CPUs Available:  1


In [87]:
test_dir = "/Users/mariappan.subramanian/Library/CloudStorage/OneDrive-TheTradeDesk/MMM/BudgetOptimizer/trash"

In [41]:
# 1. load input data
df = pd.read_csv(
    "https://raw.githubusercontent.com/google/meridian/refs/heads/main/meridian/data/simulated_data/csv/geo_media_rf.csv"
)
# 2. Create a DataFrameInputDataBuilder instance
builder = data_frame_input_data_builder.DataFrameInputDataBuilder(
    kpi_type='non_revenue'
)
builder = (
    builder.with_kpi(df, kpi_col="conversions")
    .with_revenue_per_kpi(df, revenue_per_kpi_col="revenue_per_conversion")
    .with_population(df)
    .with_controls(
        df,
        control_cols=[
            "sentiment_score_control",
            "competitor_activity_score_control",
        ],
    )
)

channels = ["Channel0", "Channel1", "Channel2"]
builder = builder.with_media(
    df,
    media_cols=[f"{channel}_impression" for channel in channels],
    media_spend_cols=[f"{channel}_spend" for channel in channels],
    media_channels=channels,
).with_reach(
    df,
    reach_cols=["Channel3_reach"],
    frequency_cols=["Channel3_frequency"],
    rf_spend_cols=["Channel3_spend"],
    rf_channels=["Channel3"],
)

data = builder.build()

### Create Meridian model input data and configuration from user provided excel file.

#### Excel file structure.
1. 'Data' sheet has the model input data for MMM
2. 'Coefficients' sheet has the geo level coefficient for different media variables
3. 'Parameters' sheet has the media parameters such as adstock, hill parameters (inflexion and shape)

#### Model config json
1. Below is a sample config

```python
    model_config = {
    
    # time and geo inputs
    'time_col': 'week',
    'geo_col': 'geo',
    'population_col': 'population',

    # kpi inputs
    'kpi_type': 'non_revenue',
    'kpi_col': 'conversions',
    'revenue_per_kpi_col': 'revenue_per_conversion',

    # impression based media inputs
    'media_cols': ['Channel0_impression', 'Channel1_impression', 'Channel2_impression'],
    'media_spend_cols': ['Channel0_spend', 'Channel1_spend', 'Channel2_spend'],
    'media_channels': ['Channel0', 'Channel1', 'Channel2'],

    # reach based media inputs
    'reach_cols': ['Channel3_reach'],
    'frequency_cols': ['Channel3_frequency'],
    'rf_spend_cols': ['Channel3_spend'],
    'rf_channels': ['Channel3'],

    # control inputs
    'control_cols': ['sentiment_score_control', 'competitor_activity_score_control']
    }
```

#### instructions
##### step1: create 'data' of type data_frame_input_data_builder.DataFrameInputDataBuilder
1. Create an input class called AdhocDataLoader inside planner folder with file_name, and model_config as arguments
2. For testing use the file located at /Users/mariappan.subramanian/Library/CloudStorage/OneDrive-TheTradeDesk/MMM/BudgetOptimizer/mmm_input_artifacts.xlsx and the model_config above
3. Parse the input data ('Data' sheet in the excel) and the model_config to create data object.
4. Here is a sample block of code on how to do it. But go through meridian utils and figure out more.

```python
builder = data_frame_input_data_builder.DataFrameInputDataBuilder(
    kpi_type='non_revenue'
)
builder = (
    builder.with_kpi(df, kpi_col="conversions")
    .with_revenue_per_kpi(df, revenue_per_kpi_col="revenue_per_conversion")
    .with_population(df)
    .with_controls(
        df,
        control_cols=[
            "sentiment_score_control",
            "competitor_activity_score_control",
        ],
    )
)

channels = ["Channel0", "Channel1", "Channel2"]
builder = builder.with_media(
    df,
    media_cols=[f"{channel}_impression" for channel in channels],
    media_spend_cols=[f"{channel}_spend" for channel in channels],
    media_channels=channels,
).with_reach(
    df,
    reach_cols=["Channel3_reach"],
    frequency_cols=["Channel3_frequency"],
    rf_spend_cols=["Channel3_spend"],
    rf_channels=["Channel3"],
)
```

In [43]:
data.media

<xarray.DataArray 'media' (geo: 20, media_time: 156, media_channel: 3)> Size: 75kB
array([[[1392518.,    3733.,  670235.],
        [ 937228.,  722210.,  745025.],
        [1286569.,  329778.,  786262.],
        ...,
        [1211774., 1173873.,       0.],
        [ 836566.,  305098.,  407998.],
        [1269842., 1263794.,  509309.]],

       [[3032312., 1231404.,  763501.],
        [2785805., 1548845., 1290322.],
        [2811866., 1705897.,  993765.],
        ...,
        [2274402., 2511346.,   14731.],
        [ 786126.,       0., 1411209.],
        [2067059., 2773346., 1458800.]],

       [[ 593030.,  438756.,  137622.],
        [ 534426.,  360286.,  118274.],
        [ 630650.,  424657.,  326189.],
        ...,
...
        ...,
        [ 249370.,  264544.,   21748.],
        [ 183341.,   49802.,  176608.],
        [ 275637.,  306384.,  106971.]],

       [[2582698.,  922861.,  653117.],
        [1424830., 1275632.,  962646.],
        [2497179., 1203092., 1632313.],
        ...,
        [2530296., 1230214.,       0.],
        [ 946695.,  351198., 1537708.],
        [2415999., 2758301.,  647724.]],

       [[2304498.,  753260.,  255735.],
        [2036626., 1433122., 1656609.],
        [2946513., 1484241., 1975205.],
        ...,
        [2387086., 1703876.,       0.],
        [ 944313., 1320312., 1500085.],
        [2571754., 2509916., 1247641.]]])
Coordinates:
  * media_time     (media_time) <U10 6kB '2021-01-25' ... '2024-01-15'
  * media_channel  (media_channel) object 24B 'Channel0' 'Channel1' 'Channel2'
  * geo            (geo) <U5 400B 'Geo0' 'Geo1' 'Geo2' ... 'Geo18' 'Geo19'

In [44]:
# Configure the model
roi_rf_mu = 0.2  # Mu for ROI prior for each RF channel.
roi_rf_sigma = 0.9  # Sigma for ROI prior for each RF channel.
prior = prior_distribution.PriorDistribution(
    roi_rf=tfp.distributions.LogNormal(
        roi_rf_mu, roi_rf_sigma, name=constants.ROI_RF
    )
)
model_spec = spec.ModelSpec(prior=prior)

mmm = model.Meridian(input_data=data, model_spec=model_spec)

I0000 00:00:1756312552.315521  399038 service.cc:148] XLA service 0x1685e2a20 initialized for platform Host (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1756312552.316199  399038 service.cc:156]   StreamExecutor device (0): Host, Default Version
I0000 00:00:1756312552.425821  399038 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


In [46]:
mmm

In [45]:
mmm.media_tensors.media_scaled

<tf.Tensor: shape=(20, 156, 3), dtype=float32, numpy=
array([[[1.2785451 , 0.00513463, 1.120945  ],
        [0.86051905, 0.9933782 , 1.2460287 ],
        [1.1812677 , 0.45359975, 1.3149961 ],
        ...,
        [1.1125944 , 1.6146271 , 0.        ],
        [0.7680959 , 0.41965318, 0.6823626 ],
        [1.1659098 , 1.7383108 , 0.85180175]],

       [[1.4431016 , 0.8779292 , 0.6618729 ],
        [1.3257871 , 1.1042486 , 1.1185699 ],
        [1.3381896 , 1.2162188 , 0.8614869 ],
        ...,
        [1.0824062 , 1.7904636 , 0.01277019],
        [0.3741237 , 0.        , 1.2233658 ],
        [0.98373   , 1.9772564 , 1.264622  ]],

       [[1.1784768 , 1.3061811 , 0.49816617],
        [1.0620182 , 1.0725751 , 0.42813   ],
        [1.2532357 , 1.2642082 , 1.1807438 ],
        ...,
        [0.96795547, 1.7814041 , 0.57951427],
        [0.5817824 , 0.81019926, 0.8585332 ],
        [0.98462814, 1.5941651 , 0.65682626]],

       ...,

       [[1.3197935 , 0.17467442, 0.14414196],
        [1.176

In [50]:
demo_model_path = '/Users/mariappan.subramanian/Documents/repo/forked/meridian/demo/saved_models'
demo_model_file = f"{demo_model_path}/demo_model_geo_all_channels.pkl"

mmm2 = model.load_mmm(demo_model_file)

In [66]:
posterior_data = mmm2.inference_data['posterior']
sample_fit = posterior_data.sel(chain=0, draw=0)
sample_fit.beta_gm.shape

(20, 3)

In [68]:
sample_fit.beta_gm.to_numpy()

array([[0.5552596 , 0.5261218 , 0.6041141 ],
       [0.5452615 , 0.52460927, 0.5296495 ],
       [0.61970764, 0.52289635, 0.5704361 ],
       [0.5810297 , 0.5185799 , 0.62276745],
       [0.58486843, 0.5146453 , 0.5874583 ],
       [0.6010976 , 0.531158  , 0.60001683],
       [0.5606291 , 0.5160124 , 0.5767337 ],
       [0.6160087 , 0.5341701 , 0.59297645],
       [0.5627139 , 0.53069323, 0.55384517],
       [0.5678924 , 0.5022697 , 0.5676123 ],
       [0.55914116, 0.5362947 , 0.60157704],
       [0.5892703 , 0.50667334, 0.51839924],
       [0.5720348 , 0.5392768 , 0.5943174 ],
       [0.5706436 , 0.5184157 , 0.5928508 ],
       [0.60204583, 0.5237032 , 0.5809822 ],
       [0.59824467, 0.51353633, 0.63421816],
       [0.57571524, 0.51766866, 0.58327013],
       [0.57893217, 0.54069287, 0.5628507 ],
       [0.57264596, 0.5287155 , 0.564279  ],
       [0.5574431 , 0.5357163 , 0.579171  ]], dtype=float32)

In [69]:
sample_fit.alpha_m

<xarray.DataArray 'alpha_m' (media_channel: 3)> Size: 12B
array([0.40226343, 0.30872327, 0.15651429], dtype=float32)
Coordinates:
    chain          int64 8B 0
    draw           int64 8B 0
  * media_channel  (media_channel) object 24B 'Channel0' 'Channel1' 'Channel2'

In [74]:
posterior_data.beta_gm.__class__


xarray.core.dataarray.DataArray

In [79]:
posterior_data.alpha_m.median(dim=('chain', 'draw')).to_pandas()

media_channel
Channel0    0.510567
Channel1    0.286708
Channel2    0.171263
Name: alpha_m, dtype: float32

In [ ]:
posterior_data.beta_gm.median(dim=('chain', 'draw')).to_pandas().reset_index()

media_channel,Channel0,Channel1,Channel2
geo,,,
Geo0,0.591570,0.576957,0.650515
Geo1,0.605447,0.586188,0.615318
Geo2,0.620543,0.520246,0.634204
Geo3,0.571194,0.564354,0.640721
Geo4,0.543899,0.557873,0.638068
Geo5,0.581583,0.563858,0.635039
Geo6,0.636543,0.547197,0.628485
Geo7,0.597153,0.551261,0.632978
Geo8,0.596000,0.559742,0.634673


In [85]:
posterior_data.beta_grf.median(dim=('chain', 'draw')).to_pandas().reset_index()

rf_channel,geo,Channel3
0,Geo0,0.542482
1,Geo1,0.518652
2,Geo2,0.547071
3,Geo3,0.557285
4,Geo4,0.529449
5,Geo5,0.537765
6,Geo6,0.523995
7,Geo7,0.531292
8,Geo8,0.548735
9,Geo9,0.542647


In [76]:
posterior_data.beta_gm

<xarray.DataArray 'beta_gm' (chain: 10, draw: 1000, geo: 20, media_channel: 3)> Size: 2MB
array([[[[0.5552596 , 0.5261218 , 0.6041141 ],
         [0.5452615 , 0.52460927, 0.5296495 ],
         [0.61970764, 0.52289635, 0.5704361 ],
         ...,
         [0.57893217, 0.54069287, 0.5628507 ],
         [0.57264596, 0.5287155 , 0.564279  ],
         [0.5574431 , 0.5357163 , 0.579171  ]],

        [[0.574977  , 0.5965901 , 0.70333827],
         [0.5534665 , 0.54458284, 0.70148087],
         [0.636431  , 0.52895373, 0.69549584],
         ...,
         [0.5233603 , 0.61177444, 0.6956042 ],
         [0.53616774, 0.5503575 , 0.7104173 ],
         [0.54760134, 0.5649491 , 0.6890485 ]],

        [[0.5717939 , 0.54426575, 0.63458925],
         [0.57451344, 0.5562644 , 0.63520825],
         [0.63002545, 0.57618487, 0.63532573],
         ...,
...
         ...,
         [0.36947572, 0.6351249 , 0.58783233],
         [0.38509113, 0.62143964, 0.63484603],
         [0.50185055, 0.6816189 , 0.62717533]],

        [[0.3785497 , 0.58288556, 0.65895087],
         [0.37201667, 0.6099496 , 0.6352708 ],
         [0.5015548 , 0.6118517 , 0.64571285],
         ...,
         [0.41483796, 0.5697418 , 0.60838604],
         [0.37321776, 0.58393854, 0.6591141 ],
         [0.4746935 , 0.6230212 , 0.6232447 ]],

        [[0.6574578 , 0.592354  , 0.5756557 ],
         [0.62887114, 0.51531243, 0.5794441 ],
         [0.75666255, 0.52897614, 0.591169  ],
         ...,
         [0.7471764 , 0.5724279 , 0.58419544],
         [0.7387186 , 0.5883835 , 0.5853315 ],
         [0.9146155 , 0.5808156 , 0.58988917]]]], dtype=float32)
Coordinates:
  * chain          (chain) int64 80B 0 1 2 3 4 5 6 7 8 9
  * draw           (draw) int64 8kB 0 1 2 3 4 5 6 ... 994 995 996 997 998 999
  * media_channel  (media_channel) object 24B 'Channel0' 'Channel1' 'Channel2'
  * geo            (geo) <U5 400B 'Geo0' 'Geo1' 'Geo2' ... 'Geo18' 'Geo19'

In [52]:
inference_data_pos = mmm2.inference_data.groups()['posterior']
inference_data_pos

TypeError: list indices must be integers or slices, not str

In [49]:
mmm.inference_data.groups()

[]

In [47]:
analyzer_object = analyzer.Analyzer(mmm)

In [ ]:
analyzer_object.

In [34]:
# optimization user inputs
# spend_constraint_default
# fixed_budget or target_roi or target_mroi

## Independent Budget Optimizer Framework

### Idea: 

Develop a budget optimizer that works with the same logic as Meridian's budget optimizer (BudgetOptimizer class in @meridian/analysis/optimizer.py) but does not use Meridian's fitted model. Instead it would get all the inputs required to run the optimizer such as media transformation parameters (adstock, hill function parameters) and media coefficients via an excel file. Reuse the inbuilt meridian functions wherever necessary.

### Sample input file (user input)

1. The sample input file with the inputs to run the optimizer can be found at @meridian/data/simulated_data/xlsx/independent_optimizer_input_file.xlsx
2. The first sheet named 'data' has the historical granular data

### Sample `model_config` (user input)
```python
    model_config = {
    
    # time and geo inputs
    'time_col': 'week',
    'geo_col': 'geo',
    'population_col': 'population',

    # kpi inputs
    'kpi_type': 'non_revenue',
    'kpi_col': 'conversions',
    'revenue_per_kpi_col': 'revenue_per_conversion',

    # impression based media inputs
    'media_cols': ['Channel0_impression', 'Channel1_impression', 'Channel2_impression'],
    'media_spend_cols': ['Channel0_spend', 'Channel1_spend', 'Channel2_spend'],
    'media_channels': ['Channel0', 'Channel1', 'Channel2'],

    # reach based media inputs
    'reach_cols': ['Channel3_reach'],
    'frequency_cols': ['Channel3_frequency'],
    'rf_spend_cols': ['Channel3_spend'],
    'rf_channels': ['Channel3'],

    # control inputs
    'control_cols': ['sentiment_score_control', 'competitor_activity_score_control']
    }
```

#### Instructions

1. 

2. Create an IndependentBudgetOptimizer class based off of BudgetOptimizer class in @meridian/analysis/optimizer.py and make sure you make changes only in those places where meridian's fitted model object is referenced. Otherwise, keep the codes from the existing framework as is.

2. input data from data sheet should be converted 

In [38]:
model_config = {

  # impression based media inputs
  'media_cols': ['Channel0_impression', 'Channel1_impression', 'Channel2_impression'],
  'media_spend_cols': ['Channel0_spend', 'Channel1_spend', 'Channel2_spend'],
  'media_channels': ['Channel0', 'Channel1', 'Channel2'],

  # reach based media inputs
  'reach_cols': ['Channel3_reach'],
  'frequency_cols': ['Channel3_frequency'],
  'rf_spend_cols': ['Channel3_spend'],
  'rf_channels': ['Channel3'],

  # kpi inputs
  'kpi_type': 'non_revenue',
  'kpi_col': 'conversions',
  'revenue_per_kpi_col': 'revenue_per_conversion',
  'population_col': 'population',

  # control inputs
  'control_cols': ['sentiment_score_control', 'competitor_activity_score_control']
}

In [35]:
from meridian.model import model
from meridian.analysis import optimizer

1. load a sample model object

In [36]:
demo_model_path = '/Users/mariappan.subramanian/Documents/repo/forked/meridian/demo/saved_models'
demo_model_file = f"{demo_model_path}/demo_model_geo_all_channels.pkl"

mmm = model.load_mmm(demo_model_file)

In [37]:
mmm.input_data

InputData(kpi=<xarray.DataArray 'kpi' (geo: 20, time: 156)> Size: 25kB
array([[12530976. ,  4926880.5, 10300557. , ..., 11532075. ,  9297790. ,
        11967258. ],
       [19017426. , 12329214. , 14298232. , ..., 20809086. , 13530469. ,
         9430961. ],
       [ 4897822.5,  4059605.8,  4581813. , ...,  5128418.5,  4869155. ,
         5191422. ],
       ...,
       [ 2883451.8,  1483594.5,  2006459.6, ...,  1643369.9,  1507497.6,
         1943231. ],
       [16736263. , 12672171. , 11119188. , ..., 16901688. , 21553568. ,
        10977840. ],
       [17849746. , 17027388. , 25654618. , ..., 22121394. , 16636714. ,
        22902830. ]])
Coordinates:
  * time     (time) <U10 6kB '2021-01-25' '2021-02-01' ... '2024-01-15'
  * geo      (geo) <U5 400B 'Geo0' 'Geo1' 'Geo2' ... 'Geo17' 'Geo18' 'Geo19', kpi_type='non_revenue', population=<xarray.DataArray 'population' (geo: 20)> Size: 160B
array([487878.  , 941246.6 , 225414.62, 833550.7 , 157739.06, 598300.25,
       227916.27, 478603.25,

In [4]:
%%time
# budget_optimizer = optimizer.BudgetOptimizer(mmm)
# optimization_results = budget_optimizer.optimize()

CPU times: user 2 μs, sys: 1 μs, total: 3 μs
Wall time: 5.96 μs


manual run

In [14]:
from collections.abc import Mapping, Sequence
import dataclasses
import functools
import math
import os
from typing import Any, TypeAlias
import warnings

import altair as alt
import jinja2
from meridian import constants as c
from meridian.analysis import analyzer
from meridian.analysis import formatter
from meridian.analysis import summary_text
from meridian.data import time_coordinates as tc
from meridian.model import model
import numpy as np
import pandas as pd
import tensorflow as tf
import xarray as xr

from meridian.analysis.optimizer import _SpendConstraint, OptimizationGrid, _validate_budget

In [5]:
# initialize
self = optimizer.BudgetOptimizer(mmm)

In [11]:
new_data: analyzer.DataTensors | None = None
use_posterior: bool = True
selected_times: tuple[str | None, str | None] | None = None
start_date: tc.Date = None
end_date: tc.Date = None
fixed_budget: bool = True
budget: float | None = None
pct_of_spend: Sequence[float] | None = None
spend_constraint_lower: _SpendConstraint | None = None
spend_constraint_upper: _SpendConstraint | None = None
target_roi: float | None = None
target_mroi: float | None = None
gtol: float = 0.0001
use_optimal_frequency: bool = True
use_kpi: bool = False
confidence_level: float = c.DEFAULT_CONFIDENCE_LEVEL
batch_size: int = c.DEFAULT_BATCH_SIZE
optimization_grid: OptimizationGrid | None = None

In [15]:
if selected_times is not None:
  warnings.warn(
      '`selected_times` is deprecated. Please use `start_date` and'
      ' `end_date` instead.',
      DeprecationWarning,
      stacklevel=2,
  )
  deprecated_start_date, deprecated_end_date = selected_times
  start_date = start_date or deprecated_start_date
  end_date = end_date or deprecated_end_date

_validate_budget(
    fixed_budget=fixed_budget,
    budget=budget,
    target_roi=target_roi,
    target_mroi=target_mroi,
)

In [19]:
spend_constraint_default = (
    c.SPEND_CONSTRAINT_DEFAULT_FIXED_BUDGET
    if fixed_budget
    else c.SPEND_CONSTRAINT_DEFAULT_FLEXIBLE_BUDGET
)

if spend_constraint_lower is None:
  spend_constraint_lower = spend_constraint_default
if spend_constraint_upper is None:
  spend_constraint_upper = spend_constraint_default


In [20]:
spend_constraint_default, spend_constraint_lower, spend_constraint_upper

(0.3, 0.3, 0.3)

In [23]:
# inside create_optimization_grid
# optimization_grid = self.create_optimization_grid(
#     new_data=new_data,
#     start_date=start_date,
#     end_date=end_date,
#     budget=budget,
#     pct_of_spend=pct_of_spend,
#     spend_constraint_lower=spend_constraint_lower,
#     spend_constraint_upper=spend_constraint_upper,
#     gtol=gtol,
#     use_posterior=use_posterior,
#     use_kpi=use_kpi,
#     use_optimal_frequency=use_optimal_frequency,
#     batch_size=batch_size,
# )
batch_size: int = c.DEFAULT_BATCH_SIZE  # default 100
if new_data is None:
  new_data = analyzer.DataTensors()


In [31]:
required_tensors = c.PERFORMANCE_DATA + (c.TIME,)
required_tensors = c.PERFORMANCE_DATA + (c.TIME,)
filled_data = new_data.validate_and_fill_missing_data(
    required_tensors_names=required_tensors, meridian=self._meridian
)

In [ ]:
#

filled_data.media

<tf.Tensor: shape=(20, 156, 3), dtype=float32, numpy=
array([[[1392518.,    3733.,  670235.],
        [ 937228.,  722210.,  745025.],
        [1286569.,  329778.,  786262.],
        ...,
        [1211774., 1173873.,       0.],
        [ 836566.,  305098.,  407998.],
        [1269842., 1263794.,  509309.]],

       [[3032312., 1231404.,  763501.],
        [2785805., 1548845., 1290322.],
        [2811866., 1705897.,  993765.],
        ...,
        [2274402., 2511346.,   14731.],
        [ 786126.,       0., 1411209.],
        [2067059., 2773346., 1458800.]],

       [[ 593030.,  438756.,  137622.],
        [ 534426.,  360286.,  118274.],
        [ 630650.,  424657.,  326189.],
        ...,
        [ 487092.,  598387.,  160095.],
        [ 292763.,  272152.,  237176.],
        [ 495482.,  535492.,  181453.]],

       ...,

       [[ 352217.,   31117.,   21118.],
        [ 313975.,  143726.,  252434.],
        [ 269650.,  244641.,   57590.],
        ...,
        [ 249370.,  264544.,   2174